# SpeciesNet Setup (third-party/eb_cameratrapai)

This notebook provides a step-by-step setup path for the local **SpeciesNet**
species classifier under:

- `third-party/eb_cameratrapai`

SpeciesNet (Google `cameratrapai`) is packaged as the `speciesnet` Python package.
It is an ensemble of an object detector, a species classifier, and geofencing
heuristics. This notebook clones and installs it for repeatable environment setup.

## Step 1 — Confirm repository root and SpeciesNet path

In [1]:
from pathlib import Path
import subprocess

repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
snet_path = repo_root / "third-party" / "eb_cameratrapai"

print(f"Repository root: {repo_root}")
print(f"SpeciesNet path: {snet_path}")
print(f"Exists: {snet_path.exists()}")

Repository root: /Users/elhorte/git/ebio/project-id
SpeciesNet path: /Users/elhorte/git/ebio/project-id/third-party/eb_cameratrapai
Exists: True


## Step 2 — Clone Earth-Biometrics SpeciesNet repo if missing

Run the next cell only if `third-party/eb_cameratrapai` does not exist.

Source repo:
- `git@github.com:Earth-Biometrics/eb_cameratrapai.git`

In [2]:
from pathlib import Path
import subprocess

repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
third_party_dir = repo_root / "third-party"
snet_dir = third_party_dir / "eb_cameratrapai"
repo_url = "git@github.com:Earth-Biometrics/eb_cameratrapai.git"

third_party_dir.mkdir(parents=True, exist_ok=True)
if snet_dir.exists():
    print(f"SpeciesNet already present at {snet_dir}")
else:
    subprocess.run(["git", "clone", repo_url, str(snet_dir)], check=True)
    print(f"Cloned {repo_url} to {snet_dir}")

SpeciesNet already present at /Users/elhorte/git/ebio/project-id/third-party/eb_cameratrapai


## Step 3 — Install SpeciesNet (editable)

Installs the `speciesnet` package in editable mode, pulling in its runtime
dependencies (torch, yolov5, onnx2torch, kagglehub, reverse_geocoder, …) declared
in `pyproject.toml`.

> On first model use (in notebook 05) SpeciesNet downloads its weights (~214 MB)
> from Kaggle via `kagglehub`. This works without a Kaggle login for the default
> public model.

> **Restart note:** an editable install registers its path via a mechanism Python
> only wires up at interpreter startup. Step 4 (and notebook 05) add the checkout
> to `sys.path` at runtime, so a kernel restart is **not required** — but
> restarting is still the cleanest option if you prefer.

In [3]:
from pathlib import Path
import subprocess
import sys

repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
snet_root = repo_root / "third-party" / "eb_cameratrapai"

if not snet_root.exists():
    raise FileNotFoundError(f"SpeciesNet repo not found: {snet_root}. Run Step 2 first.")
if not (snet_root / "pyproject.toml").exists():
    raise FileNotFoundError(
        f"pyproject.toml not found in {snet_root}; expected an eb_cameratrapai checkout."
    )

cmd = [sys.executable, "-m", "pip", "install", "-e", str(snet_root)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("\nSpeciesNet installed in editable mode.")

Running: /Users/elhorte/git/ebio/project-id/.venv/bin/python -m pip install -e /Users/elhorte/git/ebio/project-id/third-party/eb_cameratrapai
Obtaining file:///Users/elhorte/git/ebio/project-id/third-party/eb_cameratrapai
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for speciesnet (pyproject.toml): started
  Building editable for speciesnet (pyproject.toml): finished with status 'done'
  Created wheel for speciesnet: filename=speciesnet-5.0.5-0.editable-py3-none-any.whl size=16889 sha256=32e306dbe9f3c6c9b5a9d675e607fb555e82

## Step 4 — Smoke test imports

This verifies that SpeciesNet and its core dependencies import cleanly.

In [4]:
import sys
import subprocess
from pathlib import Path

# SpeciesNet is installed editable; its path is registered via a finder that
# Python only wires up at interpreter startup, so a kernel that was running
# during the Step-3 install can't see `speciesnet` until it restarts. Adding the
# checkout to sys.path makes the import work without a restart.
_snet_root = (
    Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
    / "third-party" / "eb_cameratrapai"
)
if _snet_root.is_dir() and str(_snet_root) not in sys.path:
    sys.path.insert(0, str(_snet_root))

import importlib

modules = ["torch", "numpy", "PIL", "yolov5", "speciesnet"]
results = {}
for name in modules:
    try:
        importlib.import_module(name)
        results[name] = "OK"
    except Exception as exc:
        results[name] = f"FAILED: {exc}"

# Confirm the key SpeciesNet entry points are importable.
try:
    from speciesnet import SpeciesNet, DEFAULT_MODEL, draw_bboxes, load_rgb_image
    results["SpeciesNet"] = "OK"
    results["DEFAULT_MODEL"] = DEFAULT_MODEL
except Exception as exc:
    results["SpeciesNet"] = f"FAILED: {exc}"

if "No module named 'speciesnet'" in results.get("speciesnet", ""):
    print("Hint: speciesnet is installed but not visible to this kernel.")
    print("      Restart the kernel (Kernel -> Restart) and rerun this cell.")

results

/Users/elhorte/git/ebio/project-id/.venv/lib/python3.13/site-packages/yolov5/utils/general.py:34: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg


{'torch': 'OK',
 'numpy': 'OK',
 'PIL': 'OK',
 'yolov5': 'OK',
 'speciesnet': 'OK',
 'SpeciesNet': 'OK',
 'DEFAULT_MODEL': 'kaggle:google/speciesnet/pyTorch/v4.0.3a/1'}

## Step 5 — Next action

After successful setup, run:

- [05_speciesnet_local_images.ipynb](05_speciesnet_local_images.ipynb)

to classify local images with SpeciesNet.